# Examination Room Scheduling — Final KPI Analysis

**Best Policy Selected:** F: Admin Buffer

This notebook presents all Key Performance Indicators for all 6 scheduling policies
across both weeks, with deep-dive analysis for the selected best policy.

**Bounds computed (CG → LR → TS pipeline):**
| Bound | Week 1 | Week 2 |
|-------|--------|--------|
| z_LR (Lagrangian LB) | 33950736.90 | 39001032.90 |
| z_LP (CG LP bound)   | 33900743.38 | 38913521.92 |
| z_ILP (CG ILP)       | 33950736.90 | 39001032.90 |
| z_TS (Tabu Search)   | 30002360.80 | 34052955.30 |
| Duality Gap (%)      | -13.16% | -14.53% |


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'models'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

# ── Load pre-computed results ─────────────────────────────────────────────────
HERE = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
FINAL = os.path.join(HERE, '..', '..', 'final_results')

kpis_w1 = pd.read_csv(f'{FINAL}/policy_comparison_kpis_week1.csv')
kpis_w2 = pd.read_csv(f'{FINAL}/policy_comparison_kpis_week2.csv')
sched_w1 = pd.read_csv(f'{FINAL}/schedules/best_policy_schedule_week1.csv')
sched_w2 = pd.read_csv(f'{FINAL}/schedules/best_policy_schedule_week2.csv')
bounds_w1 = pd.read_csv(f'{FINAL}/bounds/bounds_summary_week1.csv')
bounds_w2 = pd.read_csv(f'{FINAL}/bounds/bounds_summary_week2.csv')

print("Data loaded successfully.")
print(f"  Week 1: {len(sched_w1)} appointments")
print(f"  Week 2: {len(sched_w2)} appointments")


## 1. Policy Comparison — All 6 Policies, Both Weeks

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Policy Comparison — Key Performance Indicators', fontsize=15, fontweight='bold')

metrics = [
    ('Coverage (%)',          'Coverage (%)',            True),
    ('Avg Switches/PD',       'Avg Room Switches / Provider-Day', False),
    ('Total Travel (m)',      'Total Travel Distance (m)',        False),
    ('Rooms Used',            'Distinct Rooms Used',              False),
]
weeks  = [kpis_w1, kpis_w2]
titles = ['Week 1', 'Week 2']
colors = plt.cm.Set2(np.linspace(0, 1, len(kpis_w1)))

for col_i, (metric, label, higher_better) in enumerate(metrics):
    for row_i, (df, wtitle) in enumerate(zip(weeks, titles)):
        ax = axes[row_i][col_i]
        vals = df[metric]
        bars = ax.bar(df['Policy'], vals, color=colors, edgecolor='black', linewidth=0.7)
        ax.set_title(f'{label}\n{wtitle}', fontsize=9)
        ax.set_xticklabels(df['Policy'], rotation=30, ha='right', fontsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01*(vals.max() or 1),
                    f'{v:.1f}', ha='center', va='bottom', fontsize=8)
        if higher_better:
            ax.set_ylim(0, 115)

plt.tight_layout()
plt.savefig(f'{FINAL}/kpis/policy_comparison_all.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved policy_comparison_all.png")


## 2. Full KPI Tables

In [ ]:
print("=== WEEK 1 ===")
display(kpis_w1.set_index('Policy'))
print("\n=== WEEK 2 ===")
display(kpis_w2.set_index('Policy'))


## 3. Best Policy Deep-Dive: F: Admin Buffer

In [ ]:
# Combine both weeks for deep-dive
sched_all = pd.concat([sched_w1, sched_w2], ignore_index=True)
scheduled = sched_all[sched_all['assigned_room'].notna()].copy()

# ── 3a: Coverage by day and week ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
colors_days = plt.cm.Pastel1(np.linspace(0, 1, 5))

for ax, (df, label) in zip(axes, [(sched_w1, 'Week 1'), (sched_w2, 'Week 2')]):
    day_cov = []
    for d in days_order:
        ddf = df[df['day_of_week'] == d]
        if len(ddf) == 0:
            day_cov.append(0)
        else:
            day_cov.append(ddf['assigned_room'].notna().mean() * 100)
    bars = ax.bar(days_order, day_cov, color=colors_days, edgecolor='black', linewidth=0.7)
    ax.set_title(f'Coverage by Day — {label}', fontsize=11)
    ax.set_ylim(0, 110)
    ax.set_ylabel('Coverage (%)')
    for bar, v in zip(bars, day_cov):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{v:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FINAL}/kpis/coverage_by_day.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3b: Switches and travel per provider-day ─────────────────────────────────
if 'num_switches' in sched_all.columns and 'total_travel' in sched_all.columns:
    pd_stats_all = (sched_all
        .groupby(['provider', 'day_of_week', 'week'])
        .agg(num_switches=('num_switches', 'first'),
             total_travel=('total_travel', 'first'))
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Switches distribution
    ax = axes[0]
    w1_sw = pd_stats_all[pd_stats_all['week'] == 1]['num_switches']
    w2_sw = pd_stats_all[pd_stats_all['week'] == 2]['num_switches']
    ax.hist([w1_sw, w2_sw], bins=range(0, int(max(w1_sw.max(), w2_sw.max())) + 2),
            label=['Week 1', 'Week 2'], color=['steelblue', 'coral'],
            edgecolor='black', alpha=0.8, rwidth=0.8)
    ax.set_title('Room Switches per Provider-Day Distribution', fontsize=11)
    ax.set_xlabel('Number of Room Switches')
    ax.set_ylabel('Count of Provider-Days')
    ax.legend()
    ax.set_xticks(range(0, int(max(w1_sw.max(), w2_sw.max())) + 2))

    # Travel distribution
    ax = axes[1]
    w1_tr = pd_stats_all[pd_stats_all['week'] == 1]['total_travel']
    w2_tr = pd_stats_all[pd_stats_all['week'] == 2]['total_travel']
    max_tr = max(w1_tr.max(), w2_tr.max())
    bins = np.linspace(0, max_tr + 1, 20)
    ax.hist([w1_tr, w2_tr], bins=bins, label=['Week 1', 'Week 2'],
            color=['steelblue', 'coral'], edgecolor='black', alpha=0.8)
    ax.set_title('Total Travel per Provider-Day Distribution', fontsize=11)
    ax.set_xlabel('Travel Distance (m)')
    ax.set_ylabel('Count of Provider-Days')
    ax.legend()

    plt.tight_layout()
    plt.savefig(f'{FINAL}/kpis/switches_travel_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── 3c: Room utilisation heatmap (rooms × days) ───────────────────────────────
for week_num, sched in [(1, sched_w1), (2, sched_w2)]:
    sched_ok = sched[sched['assigned_room'].notna()].copy()
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    # minutes used per (room, day)
    room_day_min = (sched_ok.groupby(['assigned_room', 'day_of_week'])
                    ['duration_min'].sum()
                    .unstack(fill_value=0)
                    .reindex(columns=[d for d in days_order if d in sched_ok['day_of_week'].unique()],
                             fill_value=0))

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(room_day_min.values, aspect='auto', cmap='YlOrRd')
    plt.colorbar(im, ax=ax, label='Occupied Minutes')
    ax.set_xticks(range(len(room_day_min.columns)))
    ax.set_xticklabels(room_day_min.columns, fontsize=9)
    ax.set_yticks(range(len(room_day_min.index)))
    ax.set_yticklabels(room_day_min.index, fontsize=9)
    ax.set_title(f'Room Utilisation Heatmap — Week {week_num}', fontsize=12)
    for i in range(len(room_day_min.index)):
        for j in range(len(room_day_min.columns)):
            v = room_day_min.values[i, j]
            ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                    fontsize=8, color='black' if v < room_day_min.values.max()*0.6 else 'white')
    plt.tight_layout()
    plt.savefig(f'{FINAL}/kpis/room_utilisation_heatmap_week{week_num}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── 3d: Provider compactness by provider ─────────────────────────────────────
for week_num, sched in [(1, sched_w1), (2, sched_w2)]:
    sched_ok = sched[sched['assigned_room'].notna()].copy()
    rooms_per_pd = (sched_ok.groupby(['provider', 'day_of_week'])
                   ['assigned_room'].nunique().reset_index())
    rooms_per_pd.columns = ['provider', 'day', 'rooms_used']
    provider_compact = (rooms_per_pd.groupby('provider')
                        .apply(lambda g: (g['rooms_used'] == 1).mean() * 100)
                        .sort_values(ascending=False))

    fig, ax = plt.subplots(figsize=(16, 5))
    colors_compact = ['#2ecc71' if v == 100 else '#e74c3c' if v < 50 else '#f39c12'
                      for v in provider_compact.values]
    ax.bar(provider_compact.index, provider_compact.values,
           color=colors_compact, edgecolor='black', linewidth=0.6)
    ax.axhline(100, color='green', linestyle='--', linewidth=1, alpha=0.5, label='100% compact')
    ax.set_title(f'Provider Compactness (% of days in single room) — Week {week_num}', fontsize=11)
    ax.set_xlabel('Provider')
    ax.set_ylabel('Compactness (%)')
    ax.set_xticklabels(provider_compact.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 115)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FINAL}/kpis/provider_compactness_week{week_num}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── 3e: Peak concurrent rooms by time-of-day ─────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(25, 10), sharey=True)
fig.suptitle('Concurrent Rooms in Use by Time of Day', fontsize=13, fontweight='bold')

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
for row_i, (sched, wlabel) in enumerate([(sched_w1, 'Week 1'), (sched_w2, 'Week 2')]):
    sched_ok = sched[sched['assigned_room'].notna()].copy()
    for col_i, day in enumerate(days_order):
        ax = axes[row_i][col_i]
        day_df = sched_ok[sched_ok['day_of_week'] == day]
        if day_df.empty:
            ax.set_title(f'{day}\n{wlabel}', fontsize=8)
            continue
        time_range = range(int(day_df['start_min'].min()), int(day_df['end_min'].max()) + 1)
        concurrent = []
        times = []
        for t in time_range:
            n = ((day_df['start_min'] <= t) & (day_df['end_min'] > t)).sum()
            concurrent.append(n)
            times.append(t)
        ax.fill_between(times, concurrent, alpha=0.6, color='steelblue')
        ax.plot(times, concurrent, color='navy', linewidth=0.8)
        ax.axvline(720, color='red', linestyle='--', linewidth=0.8, alpha=0.5)  # lunch
        ax.set_title(f'{day}\n{wlabel}', fontsize=8)
        ax.set_xlabel('Time (min)', fontsize=7)
        if col_i == 0:
            ax.set_ylabel('Concurrent Rooms', fontsize=8)
        x_ticks = range(480, 1021, 120)
        ax.set_xticks(list(x_ticks))
        ax.set_xticklabels([f'{t//60:02d}:{t%60:02d}' for t in x_ticks], fontsize=6, rotation=45)

plt.tight_layout()
plt.savefig(f'{FINAL}/kpis/concurrent_rooms_by_time.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Optimality Bounds (CG → LR → TS Pipeline)

In [ ]:
# Bounds comparison
bounds_combined = pd.concat([
    bounds_w1.assign(Week='Week 1'),
    bounds_w2.assign(Week='Week 2'),
])
print("Bounds Summary:")
display(bounds_combined.set_index('Week'))

# Plot bounds hierarchy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimality Bounds — CG → LR → TS Pipeline', fontsize=13, fontweight='bold')

bound_cols = ['z_LR (Lagrangian LB)', 'z_LP (CG LP bound)', 'z_ILP (CG ILP)', 'z_TS (Tabu Search)']
labels     = ['z_LR\n(Lagrangian LB)', 'z_LP\n(CG LP bound)', 'z_ILP\n(CG ILP)', 'z_TS\n(Tabu Search)']
colors_bnd = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for ax, (bdf, wlabel) in zip(axes, [(bounds_w1, 'Week 1'), (bounds_w2, 'Week 2')]):
    vals = [bdf[c].values[0] for c in bound_cols]
    bars = ax.bar(labels, vals, color=colors_bnd, edgecolor='black', linewidth=0.8, width=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)
    gap = bdf['Duality Gap (%)'].values[0]
    ax.set_title(f'{wlabel}   |   Duality Gap = {gap:.2f}%', fontsize=11)
    ax.set_ylabel('Objective Value')
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.savefig(f'{FINAL}/kpis/bounds_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Tabu Search Convergence

In [ ]:
for week_num in [1, 2]:
    try:
        ts_hist = pd.read_csv(f'{FINAL}/bounds/ts_history_week{week_num}.csv')
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(ts_hist.index, ts_hist.iloc[:, 0] if 'cost' not in ts_hist.columns
                else ts_hist['cost'], label='Current cost', alpha=0.7, linewidth=1)
        if 'best_cost' in ts_hist.columns:
            ax.plot(ts_hist.index, ts_hist['best_cost'], label='Best cost', linewidth=1.5, color='red')
        ax.set_title(f'Tabu Search Convergence — Week {week_num}', fontsize=11)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Objective Value')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{FINAL}/kpis/ts_convergence_week{week_num}.png', dpi=150, bbox_inches='tight')
        plt.show()
    except FileNotFoundError:
        print(f"TS history for week {week_num} not found.")


## 6. Schedule Gantt Charts

Gantt charts saved in `final_results/gantt_charts/`. Preview overviews below.

In [ ]:
from IPython.display import Image, display as ipy_display
for week_num in [1, 2]:
    fpath = f'{FINAL}/gantt_charts/overview/week{week_num}_overview.png'
    if os.path.exists(fpath):
        print(f"\nWeek {week_num} Overview:")
        ipy_display(Image(filename=fpath, width=1200))


## 7. Additional KPIs

In [ ]:
# No-show appointments (if tracked)
for week_num, sched in [(1, sched_w1), (2, sched_w2)]:
    print(f"\n=== Week {week_num} ===")
    total = len(sched)
    scheduled_count = sched['assigned_room'].notna().sum()
    unscheduled = total - scheduled_count

    # Appointments per day
    day_counts = sched.groupby('day_of_week').size().reindex(
        ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'], fill_value=0)
    print(f"  Total appointments      : {total}")
    print(f"  Scheduled               : {scheduled_count} ({scheduled_count/total*100:.1f}%)")
    print(f"  Unscheduled             : {unscheduled} ({unscheduled/total*100:.1f}%)")
    print(f"  Unique providers        : {sched['provider'].nunique()}")
    print(f"  Unique rooms used       : {sched['assigned_room'].nunique()}")
    print(f"  Appointments by day:")
    for day, cnt in day_counts.items():
        print(f"    {day:<12}: {cnt}")


In [ ]:
# Avg appointments per provider per day
for week_num, sched in [(1, sched_w1), (2, sched_w2)]:
    sched_ok = sched[sched['assigned_room'].notna()]
    appts_per_pd = sched_ok.groupby(['provider', 'day_of_week']).size()
    print(f"\nWeek {week_num} — Appointments per Provider-Day:")
    print(f"  Mean : {appts_per_pd.mean():.1f}")
    print(f"  Max  : {appts_per_pd.max()}")
    print(f"  Min  : {appts_per_pd.min()}")

    fig, ax = plt.subplots(figsize=(12, 4))
    appts_per_pd.hist(ax=ax, bins=range(1, appts_per_pd.max() + 2),
                      color='steelblue', edgecolor='black', rwidth=0.8)
    ax.set_title(f'Appointments per Provider-Day — Week {week_num}', fontsize=11)
    ax.set_xlabel('Number of Appointments')
    ax.set_ylabel('Count of Provider-Days')
    plt.tight_layout()
    plt.savefig(f'{FINAL}/kpis/appts_per_providerday_week{week_num}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
